In [ ]:
import os
import random
import time
from dataclasses import dataclass
from typing import Optional
import subprocess
import sys
import copy as cpy
import joblib
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tyro
from torch.distributions.normal import Normal
from torch.utils.tensorboard import SummaryWriter

from cvxopt import matrix, solvers
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

import warnings
warnings.filterwarnings('ignore')

## Hyperparameters

In [ ]:
@dataclass
class Args:
    exp_name: str = "inverted_pendulum"
    """the name of this experiment"""
    seed: int = 1
    """seed of the experiment"""
    torch_deterministic: bool = True
    """if toggled, `torch.backends.cudnn.deterministic=False`"""
    cuda: bool = False #True
    """if toggled, cuda will be enabled by default"""
    capture_video: bool = False
    """whether to capture videos of the agent performances (check out `videos` folder)"""
    save_model: bool = True
    """whether to save model into the `runs/{run_name}` folder"""

    # Algorithm arguments
    env_id: str = "InvertedPendulum-v0"
    """the id of the environment"""
    total_timesteps: int = 300000 #500000
    """total timesteps of the experiments"""
    learning_rate: float = 3e-4
    """the learning rate of the optimizer"""
    num_envs: int = 1
    """the number of parallel game environments"""
    num_steps: int = 3000
    """the number of steps to run in each environment per policy rollout"""
    anneal_lr: bool = True
    """Toggle learning rate annealing for policy and value networks"""
    gamma: float = 0.995
    """the discount factor gamma"""
    gae_lambda: float = 0.98
    """the lambda for the general advantage estimation"""
    num_minibatches: int = 10
    """the number of mini-batches"""
    update_epochs: int = 10
    """the K epochs to update the policy"""
    norm_adv: bool = True
    """Toggles advantages normalization"""
    clip_coef: float = 0.2
    """the surrogate clipping coefficient"""
    clip_vloss: bool = True
    """Toggles whether or not to use a clipped loss for the value function, as per the paper."""
    ent_coef: float = 0.0
    """coefficient of the entropy"""
    vf_coef: float = 0.5
    """coefficient of the value function"""
    max_grad_norm: float = 0.5
    """the maximum norm for the gradient clipping"""
    target_kl: Optional[float] = None
    """the target KL divergence threshold"""
    bar_lr: float = 1e-3
    """learning rate for barrier compensator"""
    bar_train_steps: int = 100
    """training iterations for barrier compensator per policy update"""
    normalize: bool = False
    """use normalization (observation/reward)"""
    checkpoint_freq: int = 50
    """save checkpoint every N iterations"""
    timestep_batch_GP_max: int = 650
    """Update GP dynamics for certain number of timesteps"""
    
    # For the runtime
    batch_size: int = 0
    """the batch size (computed in runtime)"""
    minibatch_size: int = 0
    """the mini-batch size (computed in runtime)"""
    num_iterations: int = 0
    """the number of iterations (computed in runtime)"""

## Environment

In [ ]:
class InvertedPendulumEnv(gym.Wrapper):
    def __init__(self):
        env = gym.make('Pendulum-v1', max_episode_steps=300)
        super().__init__(env)
        self.unwrapped.max_torque = 15.
        self.unwrapped.max_speed = 60.
        self.unwrapped.action_space = gym.spaces.Box(low=-self.unwrapped.max_torque, high=self.unwrapped.max_torque, shape=(1,))
        high = np.array([1., 1., self.unwrapped.max_speed])
        self.unwrapped.observation_space = gym.spaces.Box(low=-high, high=high)
        self.action_space = self.unwrapped.action_space
        self.observation_space = self.unwrapped.observation_space

    def reset(self, **kwargs):
        while True:
            obs, info = self.env.reset(**kwargs)
            if abs(self.unwrapped.state[0]) <= 1.0:
                return obs, info
            kwargs.pop('seed', None)

gym.register(id="InvertedPendulum-v0", entry_point=lambda: InvertedPendulumEnv())


def make_env(env_id, idx, capture_video, run_name, gamma, normalize=True):
    def thunk():
        if capture_video and idx == 0:
            env = gym.make(env_id, render_mode="rgb_array")
            env = gym.wrappers.RecordVideo(env, f"videos/{run_name}")
        else:
            env = gym.make(env_id)

        env = gym.wrappers.FlattenObservation(env)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env = gym.wrappers.ClipAction(env)

        if normalize:
            env = gym.wrappers.NormalizeObservation(env)
            env = gym.wrappers.TransformObservation(env, lambda obs: np.clip(obs, -10, 10))
            env = gym.wrappers.NormalizeReward(env, gamma=gamma)
            env = gym.wrappers.TransformReward(env, lambda reward: np.clip(reward, -10, 10))

        return env
    return thunk

## Gaussian Process (GP)

In [ ]:
# Nominal dynamics (wrong model)
def get_nominal_dynamics(obs, u_rl):
    dt = 0.05
    G = 10
    m = 2
    l = 2
    obs = np.squeeze(obs)
    theta = np.arctan2(obs[1], obs[0])
    theta_dot = obs[2]
    f = np.array([-3*G/(2*l)*np.sin(theta + np.pi)*dt**2 + theta_dot*dt + theta + 3/(m*l**2)*u_rl*dt**2, theta_dot - 3*G/(2*l)*np.sin(theta + np.pi)*dt + 3/(m*l**2)*u_rl*dt])
    g = np.array([3/(m*l**2)*dt**2, 3/(m*l**2)*dt])
    
    x = np.array([theta, theta_dot])
    return [np.squeeze(f), np.squeeze(g), np.squeeze(x)]

# GP Model
def build_GP_model(obs_size):
    GP_list = []
    noise = 0.01
    for i in range(obs_size-1):
        kern = C(1.0, (1e-3, 1e3)) * RBF(10, (1e-2, 1e2))
        gp = GaussianProcessRegressor(kernel=kern, alpha = noise, n_restarts_optimizer=10)
        GP_list.append(gp)
    return GP_list

# Update GP dynamics
def update_GP_dynamics(GP_model, X_obs, U_control, obs_size):
    L = X_obs.shape[0]
    err = np.zeros((L-1,obs_size-1))
    S = np.zeros((L-1,2))
    for i in range(L-1):
        [f,g,x1] = get_nominal_dynamics(X_obs[i,:],U_control[i])
        theta_p = np.arctan2(X_obs[i,1], X_obs[i,0])
        theta_dot_p = X_obs[i,2]
        theta = np.arctan2(X_obs[i+1,1], X_obs[i+1,0])
        theta_dot = X_obs[i+1,2]
        S[i,:] = np.array([theta_p, theta_dot_p])
        err[i,:] = np.array([theta, theta_dot]) - f
    GP_model[0].fit(S,err[:,0])
    GP_model[1].fit(S,err[:,1])

# GP dynamics (inference)
def get_GP_dynamics(GP_model, obs, u_rl):
    u_rl = 0
    dt = 0.05
    G = 10
    m = 2
    l = 2
    obs = np.squeeze(obs)
    theta = np.arctan2(obs[1], obs[0])
    theta_dot = obs[2]
    x = np.array([theta, theta_dot])
    f_nom = np.array([-3*G/(2*l)*np.sin(theta + np.pi)*dt**2 + theta_dot*dt + theta + 3/(m*l**2)*u_rl*dt**2, theta_dot - 3*G/(2*l)*np.sin(theta + np.pi)*dt + 3/(m*l**2)*u_rl*dt])
    g = np.array([3/(m*l**2)*dt**2, 3/(m*l**2)*dt])
    f_nom = np.squeeze(f_nom)
    f = np.zeros(2)
    [m1, std1] = GP_model[0].predict(x.reshape(1,-1), return_std=True)
    [m2, std2] = GP_model[1].predict(x.reshape(1,-1), return_std=True)
    f[0] = f_nom[0] + m1.flat[0]
    f[1] = f_nom[1] + m2.flat[0]
    return [np.squeeze(f), np.squeeze(g), np.squeeze(x), np.array([np.squeeze(std1), np.squeeze(std2)])]


## CBF Model

In [ ]:
# Barrier function model
def build_barrier(action_size):
    P = matrix(np.diag([1., 1e16]), tc='d')
    q = matrix(np.zeros(action_size+1))
    H1 = np.array([1, 0.01])
    H2 = np.array([1, -0.01])
    H3 = np.array([-1, 0.01])
    H4 = np.array([-1, -0.01])
    F = 1
    return P, q, H1, H2, H3, H4, F

# Get compensatory action based on barrier function
def control_barrier(u_rl, f, g, x, std, P, q, H1, H2, H3, H4, F, torque_bound, max_speed):
    gamma_b = 0.5
    kd = 1.5

    # Set up QP to satisfy CBF
    G = np.array([[-np.dot(H1,g), -np.dot(H2,g), -np.dot(H3,g), -np.dot(H4,g), \
                   1, -1, g[1], -g[1]], [-1, -1, -1, -1, 0, 0, 0, 0]])
    G = np.transpose(G)
    h = np.array([gamma_b*F + np.dot(H1,f) + np.dot(H1,g)*u_rl - (1-gamma_b)*np.dot(H1,x) - kd*np.abs(np.dot(H1,std)),
                  gamma_b*F + np.dot(H2,f) + np.dot(H2,g)*u_rl - (1-gamma_b)*np.dot(H2,x) - kd*np.abs(np.dot(H2,std)),
                  gamma_b*F + np.dot(H3,f) + np.dot(H3,g)*u_rl - (1-gamma_b)*np.dot(H3,x) - kd*np.abs(np.dot(H3,std)),
                  gamma_b*F + np.dot(H4,f) + np.dot(H4,g)*u_rl - (1-gamma_b)*np.dot(H4,x) - kd*np.abs(np.dot(H4,std)),
                  -u_rl + torque_bound,
                  u_rl + torque_bound,
                  -f[1] - g[1]*u_rl + max_speed,
                  f[1] + g[1]*u_rl + max_speed])
    h = np.squeeze(h).astype(np.double)
    
    # Solve QP
    G = matrix(G,tc='d')
    h = matrix(h,tc='d')
    solvers.options['show_progress'] = False
    sol = solvers.qp(P, q, G, h)
    u_bar = sol['x']

    if (np.add(np.squeeze(u_rl), np.squeeze(u_bar[0])) - 0.001 >= torque_bound):
        u_bar[0] = torque_bound - u_rl
        print("Error in QP")
    elif (np.add(np.squeeze(u_rl), np.squeeze(u_bar[0])) + 0.001 <= -torque_bound):
        u_bar[0] = -torque_bound - u_rl
        print("Error in QP")
    else:
        pass

    return np.expand_dims(np.array(u_bar[0]), 0)

## CBF Compensator Neural Network

In [ ]:
class BarrierCompensator_nn(nn.Module):
    # Neural Network Architecture
    def __init__(self, obs_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, 50),
            nn.ReLU(),
            nn.Linear(50, 40),
            nn.ReLU(),
            nn.Linear(40, action_size))
        
    def forward(self, x):
        return self.net(x)

    # Control generated by the CBF Neural Network
    def get_action(self, obs):
        with torch.no_grad():
            u_BAR = self.forward(torch.FloatTensor(obs))
        return u_BAR.numpy()

## Agent: Actor-Critic (Neural Networks)

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self, envs):
        super().__init__()
        # Critic
        # Input: State
        # Output: V(s)
        self.critic = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )

        # Actor
        # Input: State
        # Output: Action_Mean(s)
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, np.prod(envs.single_action_space.shape)), std=0.01),
        )
        self.actor_logstd = nn.Parameter(torch.zeros(1, np.prod(envs.single_action_space.shape)))

    def get_value(self, x):
        return self.critic(x)

    # Sample action and compute V(s)
    def get_action_and_value(self, x, action=None):
        action_mean = self.actor_mean(x)
        action_logstd = self.actor_logstd.expand_as(action_mean)
        action_std = torch.exp(action_logstd)
        probs = Normal(action_mean, action_std)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action).sum(1), probs.entropy().sum(1), self.critic(x)

## Experiment Arguments and Logs settings

In [ ]:
args = Args()
args.batch_size = int(args.num_envs * args.num_steps)
args.minibatch_size = int(args.batch_size // args.num_minibatches)
args.num_iterations = args.total_timesteps // args.batch_size
run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

writer = SummaryWriter(f"runs/{run_name}")
writer.add_text(
    "hyperparameters",
    "|param|value|\n|-|-|\n%s" % ("\n".join([f"|{key}|{value}|" for key, value in vars(args).items()])),
)

# CSV Logging setup
csv_path_episodes = f"runs/{run_name}/log_episodes.csv"
csv_path_iters = f"runs/{run_name}/log_iters.csv"

log_episodes = []
log_iters = []

## Environment Setup

In [ ]:
# Seeding
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

# Environment
envs = gym.vector.SyncVectorEnv(
    [make_env(args.env_id, i, args.capture_video, run_name, args.gamma, normalize=args.normalize) for i in range(args.num_envs)]
)
assert isinstance(envs.single_action_space, gym.spaces.Box), "only continuous action space is supported"

# Agent and Optimizer
agent = Agent(envs).to(device)
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)

# CBF Parameters
torque_bound = 15.
max_speed = 60.
action_size = int(np.prod(envs.single_action_space.shape))
obs_size = int(np.array(envs.single_observation_space.shape).prod())
P, q, H1, H2, H3, H4, F = build_barrier(action_size)

# Barrier Compensator
bar_comp_nn = BarrierCompensator_nn(obs_size, action_size).to(device)
bar_comp_nn_optimizer = optim.Adam(bar_comp_nn.parameters(), lr=args.bar_lr)

# Build GP model of dynamics
GP_model = build_GP_model(obs_size)
GP_model_prev =  None
firstIter = True

## Storage Setup and Training Initialization

In [ ]:
# Storage Setup
obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
values = torch.zeros((args.num_steps, args.num_envs)).to(device)
action_bar_arr = np.zeros((args.num_steps, action_size))
action_BAR_arr = np.zeros((args.num_steps, action_size))
episode_obs_list = []
episode_action_list = []

# For episode tracking
episode_theta_list = []
episode_ucbf_list = []
episode_safe_list = []
ep_max_angle = 0.0
ep_mean_ucbf = 0.0
ep_max_ucbf = 0.0
ep_violations = 0

# Initialization
global_step = 0
start_time = time.time()
next_obs, _ = envs.reset(seed=args.seed)
next_obs = torch.Tensor(next_obs).to(device)
next_done = torch.zeros(args.num_envs).to(device)

## Training

In [ ]:
# Policy Training
for iteration in range(1, args.num_iterations + 1):
    # Learning rate Annealing
    if args.anneal_lr:
        frac = 1.0 - (iteration - 1.0) / args.num_iterations
        lrnow = frac * args.learning_rate
        optimizer.param_groups[0]["lr"] = lrnow

    # GP first iteration check
    if not firstIter:
        GP_model_prev = cpy.copy(GP_model)
        GP_model = build_GP_model(obs_size)
    timesteps_in_batch = 0
    
    ###########################################
    # Rollout and collect data
    ###########################################
    for step in range(0, args.num_steps):
        global_step += args.num_envs
        obs[step] = next_obs
        dones[step] = next_done

        ###########################################
        # RL-CBF-GP guided
        ###########################################
        # Sample action from RL policy
        with torch.no_grad():
            action_rl, logprob, _, value = agent.get_action_and_value(next_obs)
            values[step] = value.flatten()
        actions[step] = action_rl
        logprobs[step] = logprob

        # CBF-Neural-Network output
        prev_obs_expanded = np.expand_dims(np.squeeze(next_obs[0].cpu().numpy()), 0)
        u_BAR_ = bar_comp_nn.get_action(prev_obs_expanded)

        # RL control compensated by the CBF-Neural-Network output
        u_RL = float(action_rl.cpu().numpy()[0, 0])
        u_RL_comp = u_RL + float(u_BAR_[0, 0])

        # Dynamics with GP included
        if firstIter:
            [f,g,x,std] = get_GP_dynamics(GP_model, prev_obs_expanded, u_RL_comp)
        else:
            [f,g,x,std] = get_GP_dynamics(GP_model_prev, prev_obs_expanded, u_RL_comp)
        
        # Apply CBF correction (filter)
        u_bar_ = control_barrier(u_RL_comp, f, g, x, std, P, q, H1, H2, H3, H4, F, torque_bound, max_speed)
        u_k = u_RL_comp + u_bar_[0]
        u_k_np = np.array([[u_k]], dtype=np.float32)

        # Saving data for logs
        theta_now = x[0]
        omega_now = x[1]
        episode_theta_list.append(abs(np.degrees(theta_now)))
        episode_ucbf_list.append(abs(float(u_bar_[0])))
        episode_safe_list.append(int(abs(theta_now) + 0.01 * abs(omega_now) > F))

        # Store for compensator training
        action_bar_arr[step] = u_bar_
        action_BAR_arr[step] = u_BAR_[0]
        episode_obs_list.append(np.squeeze(next_obs[0].cpu().numpy()))
        episode_action_list.append(u_k)

        # Agent step with CBF-corrected action
        next_obs, reward, terminations, truncations, infos = envs.step(u_k_np)
        next_done = np.logical_or(terminations, truncations)
        rewards[step] = torch.tensor(reward).to(device).view(-1)
        next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(next_done).to(device)

        # End of Episode
        if terminations[0] or truncations[0]:
            if len(episode_obs_list) > 1:
                episode_obs_array = np.array(episode_obs_list)
                episode_action_array = np.array(episode_action_list)

                # GP update
                if timesteps_in_batch < args.timestep_batch_GP_max:
                    update_GP_dynamics(GP_model, episode_obs_array, episode_action_array, obs_size)

            ep_max_angle = max(episode_theta_list) if episode_theta_list else 0.0
            ep_mean_ucbf = float(np.mean(episode_ucbf_list)) if episode_ucbf_list else 0.0
            ep_max_ucbf = float(np.max(episode_ucbf_list)) if episode_ucbf_list else 0.0
            ep_violations = int(sum(episode_safe_list)) if episode_safe_list else 0
            
            timesteps_in_batch += len(episode_obs_list)
            episode_obs_list = []
            episode_action_list = []

            episode_theta_list = []
            episode_ucbf_list = []
            episode_safe_list = []

        # Logging
        if "final_info" in infos:
            for info in infos["final_info"]:
                if info and "episode" in info:
                    print(f"global_step={global_step}, episodic_return={info['episode']['r']}")
                    writer.add_scalar("charts/episodic_return", info["episode"]["r"], global_step)
                    writer.add_scalar("charts/episodic_length", info["episode"]["l"], global_step)
                    
                    # CSV logs
                    log_episodes.append({
                        "global_step": global_step,
                        "iteration": iteration,
                        "episode_return": round(float(info["episode"]["r"].item()), 4),
                        "episode_length": int(info["episode"]["l"].item()),
                        "max_angle_deg": round(ep_max_angle, 3),
                        "safety_violations": ep_violations,
                        "mean_u_cbf": round(ep_mean_ucbf, 6),
                        "max_u_cbf": round(ep_max_ucbf, 4),
                    })
    
    firstIter =  False
    
    ###########################################
    # Train CBF Compensator Neural Network
    ###########################################
    # Observation and Target
    obs_udeployed = obs.reshape((-1,) + envs.single_observation_space.shape).detach()
    correction_cbf2_target = torch.FloatTensor(action_bar_arr + action_BAR_arr).to(device)

    # Training
    for _ in range(args.bar_train_steps):
        correction_deployed = bar_comp_nn(obs_udeployed)
        comp_loss = nn.functional.mse_loss(correction_deployed, correction_cbf2_target)
        bar_comp_nn_optimizer.zero_grad()
        comp_loss.backward()
        bar_comp_nn_optimizer.step()

    writer.add_scalar("losses/compensator_loss", comp_loss.item(), global_step)

    ###########################################
    # Compute GAE
    ###########################################
    with torch.no_grad():
        next_value = agent.get_value(next_obs).reshape(1, -1)
        advantages = torch.zeros_like(rewards).to(device)
        lastgaelam = 0
        for t in reversed(range(args.num_steps)):
            if t == args.num_steps - 1:
                nextnonterminal = 1.0 - next_done
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - dones[t + 1]
                nextvalues = values[t + 1]
            delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
            advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
        returns = advantages + values

    # Flatten the batch
    b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
    b_logprobs = logprobs.reshape(-1)
    b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
    b_advantages = advantages.reshape(-1)
    b_returns = returns.reshape(-1)
    b_values = values.reshape(-1)

    ###########################################    
    # Optimizing the Policy and Value network
    ###########################################
    b_inds = np.arange(args.batch_size)
    clipfracs = []
    for epoch in range(args.update_epochs):
        np.random.shuffle(b_inds)

        # Batch iteration
        for start in range(0, args.batch_size, args.minibatch_size):
            # Minibatch
            end = start + args.minibatch_size
            mb_inds = b_inds[start:end]

            # Policies Ratio
            _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], b_actions[mb_inds])
            logratio = newlogprob - b_logprobs[mb_inds]
            ratio = logratio.exp()

            # KL divergence approx.
            with torch.no_grad():
                old_approx_kl = (-logratio).mean()
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs += [((ratio - 1.0).abs() > args.clip_coef).float().mean().item()]

            # Advantage normalization
            mb_advantages = b_advantages[mb_inds]
            if args.norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            # Policy loss (Clipped PPO)
            pg_loss1 = -mb_advantages * ratio
            pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()

            # Value loss
            newvalue = newvalue.view(-1)
            if args.clip_vloss:
                v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                v_clipped = b_values[mb_inds] + torch.clamp(
                    newvalue - b_values[mb_inds],
                    -args.clip_coef,
                    args.clip_coef,
                )
                v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
                v_loss = 0.5 * v_loss_max.mean()
            else:
                v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

            # Entropy
            entropy_loss = entropy.mean()

            # Total loss
            loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
            optimizer.step()

        # Early stopping
        if args.target_kl is not None and approx_kl > args.target_kl:
            break

    # Logging
    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

    # csv log
    u_cbf_batch = np.abs(action_bar_arr)
    mean_u_cbf_batch = float(u_cbf_batch.mean())
    max_u_cbf_batch = float(u_cbf_batch.max())
    cbf_active_frac = float((u_cbf_batch > 1e-4).mean())
    writer.add_scalar("safety/mean_u_cbf", mean_u_cbf_batch, global_step)
    writer.add_scalar("safety/cbf_active_frac", cbf_active_frac, global_step)

    log_iters.append({
        "global_step": global_step,
        "iteration": iteration,
        "policy_loss": round(pg_loss.item(),   6),
        "value_loss": round(v_loss.item(),    6),
        "compensator_loss": round(comp_loss.item(), 6),
        "mean_u_cbf_batch": round(mean_u_cbf_batch, 6),
        "max_u_cbf_batch": round(max_u_cbf_batch,  4),
        "cbf_active_frac": round(cbf_active_frac,  4),
        "explained_variance": round(explained_var,  4),
        "approx_kl": round(approx_kl.item(), 6),
    })
    
    # Record rewards
    writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
    writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
    writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
    writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
    writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
    writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
    writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
    writer.add_scalar("losses/explained_variance", explained_var, global_step)
    writer.add_scalar("charts/SPS", int(global_step / (time.time() - start_time)), global_step)

## Save model

In [ ]:
model_path = f"runs/{run_name}/{args.exp_name}.pt"
gp_path = f"runs/{run_name}/{args.exp_name}_gp.joblib"

# Save PPO Policy and CBF NN Compensator
torch.save({
    "agent": agent.state_dict(),
    "bar_comp_nn": bar_comp_nn.state_dict(),
}, model_path)

# Save GP
joblib.dump(GP_model, gp_path)
print(f"Models saved to runs/{run_name}/")

pd.DataFrame(log_episodes).to_csv(csv_path_episodes, index=False)
pd.DataFrame(log_iters).to_csv(csv_path_iters, index=False)
print(f"Logs saved to runs/{run_name}/")

envs.close()
writer.close()

## TensorBoard

In [ ]:
## TensorBoard (training plots) running at: http://localhost:6006
# subprocess.Popen([sys.executable, "-m", "tensorboard.main", "--logdir", "runs", "--port", "6006"])

In [ ]:
## Stop Tensorboard
# subprocess.run(["pkill", "-f", "tensorboard"], capture_output=True)

## Plots

In [ ]:
ep  = pd.read_csv(csv_path_episodes)
itr = pd.read_csv(csv_path_iters)

def smooth(s, w=20):
    return s.rolling(w, min_periods=1, center=True).mean()

def plot_ep(ax, col, title, ylabel, limit=None):
    ax.plot(ep.global_step, ep[col], alpha=0.2, linewidth=0.8)
    ax.plot(ep.global_step, smooth(ep[col]), linewidth=2)
    if limit:
        ax.axhline(limit, color="red", linestyle="--", linewidth=1.2, label="safe limit")
        ax.legend(fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=4))

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
fig.suptitle("PPO-CBF-GP Training", fontsize=13)
fig.subplots_adjust(hspace=0.4, wspace=0.35)

plot_ep(axes[0,0], "episode_return", "Episode Return", "Return")
plot_ep(axes[0,1], "mean_u_cbf", "Mean u_CBF (vanishing)", "u_CBF")
plot_ep(axes[1,0], "max_angle_deg", "Max Angle per Episode", "Degrees", limit=np.degrees(F))
plot_ep(axes[1,1], "safety_violations", "Safety Violations", "Steps")

fig.savefig(f"runs/{run_name}/training_curves.png", bbox_inches="tight")
plt.show()